# 🌐 Bhasha Setu — AI Video Dubbing

**One-click launch** on Google Colab with a free GPU and a public shareable link.

### How to use
1. **Set GPU runtime**: Go to `Runtime → Change runtime type → T4 GPU`
2. **Get a free ngrok token**: Sign up at [ngrok.com](https://dashboard.ngrok.com/signup) → Copy your **Authtoken** from [dashboard](https://dashboard.ngrok.com/get-started/your-authtoken)
3. **Paste your token** in the cell below
4. **Run All**: `Runtime → Run all` and wait ~3-4 minutes
5. **Click the public link** at the bottom to open Bhasha Setu!

> ⏱ First run takes ~3-4 min (model downloads). Subsequent runs start in ~30s.

---
## Step 1 — Configuration
Paste your **ngrok authtoken** below. Everything else is pre-configured.

In [ ]:
#@title 🔑 Enter your ngrok authtoken { display-mode: "form" }
#@markdown Get a free token at [ngrok.com/signup](https://dashboard.ngrok.com/signup)
NGROK_AUTHTOKEN = ""  #@param {type:"string"}

# Optional: HuggingFace token (only needed for AI Chat tab)
HF_TOKEN = ""  #@param {type:"string"}

if not NGROK_AUTHTOKEN:
    raise ValueError(
        "❌ Please enter your ngrok authtoken above!\n"
        "   Get one free at: https://dashboard.ngrok.com/signup"
    )

---
## Step 2 — Check GPU & Install System Dependencies

In [ ]:
%%bash
echo "═══════════════════════════════════════"
echo "  GPU Check"
echo "═══════════════════════════════════════"
if command -v nvidia-smi &> /dev/null; then
    nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
    echo "✅ GPU detected — dubbing will be FAST!"
else
    echo "⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU"
    echo "   (CPU mode will work but will be 10-20x slower)"
fi

echo ""
echo "═══════════════════════════════════════"
echo "  Installing ffmpeg"
echo "═══════════════════════════════════════"
apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1
ffmpeg -version | head -1
echo "✅ ffmpeg installed"

---
## Step 3 — Clone Repository & Install Python Dependencies

In [ ]:
import os

REPO_DIR = "/content/bhasha-setu"

# Clone from public GitHub repository
if not os.path.exists(REPO_DIR):
    print("📥 Cloning Bhasha Setu repository...")
    !git clone https://github.com/abhimanyugrover/Bhasha-setu-version-2.0.git {REPO_DIR}
else:
    print("✅ Repository already cloned, pulling latest...")
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f"📂 Working directory: {os.getcwd()}")

In [ ]:
# Install Python dependencies
print("📦 Installing Python dependencies (this takes ~2 minutes)...")
!pip install -q -r requirements.txt pyngrok > /dev/null 2>&1

# Verify key packages
import faster_whisper, transformers, fastapi, edge_tts
print(f"✅ faster-whisper: {faster_whisper.__version__}")
print(f"✅ transformers:   {transformers.__version__}")
print(f"✅ fastapi:        {fastapi.__version__}")
print(f"✅ edge-tts:       {edge_tts.__version__}")
print("\n🎉 All dependencies installed!")

---
## Step 4 — Pre-download ML Models
Downloads Whisper + NLLB-200 models (~750MB total). Cached for the session.

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"
print(f"🖥️  Device: {device.upper()} | Compute: {compute_type}")

# ── Pre-download Whisper model ──
print("\n📥 Downloading Whisper 'base' model...")
from faster_whisper import WhisperModel
whisper_model = WhisperModel("base", device=device, compute_type=compute_type)
del whisper_model  # Free memory — pipeline will reload it
print("✅ Whisper model cached")

# ── Pre-download NLLB-200 model ──
print("\n📥 Downloading NLLB-200 translation model...")
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
nllb_name = "facebook/nllb-200-distilled-600M"
_ = AutoTokenizer.from_pretrained(nllb_name)
_ = AutoModelForSeq2SeqLM.from_pretrained(nllb_name)
del _
print("✅ NLLB-200 model cached")

print("\n🎉 All models ready!")

---
## Step 5 — Launch Server with Public URL 🚀
This cell runs the server. **Keep it running** — the public URL stays active as long as this cell is running.

In [ ]:
import os, subprocess, time, signal
from pyngrok import ngrok, conf

os.chdir("/content/bhasha-setu")

# ── Environment setup ──
os.environ["WHISPER_DEVICE"] = "cuda" if __import__("torch").cuda.is_available() else "cpu"
os.environ["WHISPER_COMPUTE"] = "float16" if os.environ["WHISPER_DEVICE"] == "cuda" else "int8"
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

PORT = 7860

# ── Kill any previous server ──
!kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true
ngrok.kill()  # Kill any previous tunnels

# ── Configure ngrok ──
conf.get_default().auth_token = NGROK_AUTHTOKEN

# ── Start FastAPI server in background ──
print("🚀 Starting Bhasha Setu server...")
server_process = subprocess.Popen(
    [
        "uvicorn", "backend.main:app",
        "--host", "0.0.0.0",
        "--port", str(PORT),
        "--workers", "1",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    cwd="/content/bhasha-setu",
)

# ── Wait for server to be ready ──
import urllib.request, urllib.error
print("⏳ Waiting for server to start...")
for i in range(60):
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/api/languages", timeout=2)
        print(f"✅ Server is ready! (took {i+1}s)")
        break
    except (urllib.error.URLError, ConnectionRefusedError, OSError):
        time.sleep(1)
else:
    print("❌ Server failed to start. Check logs below:")
    server_process.terminate()
    print(server_process.stdout.read().decode())
    raise RuntimeError("Server startup failed")

# ── Open ngrok tunnel ──
print("\n🌐 Opening ngrok tunnel...")
public_url = ngrok.connect(PORT, "http")

print("\n" + "═" * 60)
print("  🎉 BHASHA SETU IS LIVE!")
print("═" * 60)
print(f"\n  🔗 Public URL:  {public_url}")
print(f"  🔗 Local URL:   http://localhost:{PORT}")
print(f"\n  📋 Share the public URL with anyone!")
print(f"  ⏱️  Link stays active while this cell is running.")
print("═" * 60)

# ── Stream server logs ──
print("\n📜 Server logs (Ctrl+C or stop cell to shutdown):\n")
try:
    while True:
        line = server_process.stdout.readline()
        if line:
            print(line.decode(), end="")
        elif server_process.poll() is not None:
            print("\n⚠️  Server process exited.")
            break
        else:
            time.sleep(0.1)
except KeyboardInterrupt:
    print("\n\n🛑 Shutting down...")
    server_process.terminate()
    ngrok.kill()
    print("✅ Server and tunnel stopped.")

---
## 🛑 Stop Server
Run this cell to cleanly shut down the server and tunnel.

In [ ]:
# Stop the server and ngrok tunnel
from pyngrok import ngrok
ngrok.kill()
!kill -9 $(lsof -t -i:7860) 2>/dev/null || true
print("✅ Server and tunnel stopped.")